# Step 3 — Signal Agent
**Phase 3 | NIKKO Engineering Collective**

Tests `agents/signal_agent.py`. Structured in two parts:

**Part A (no model required):** JSON extraction, enum validation, payload scrubbing, distress/risk consistency enforcement, safe fallback.

**Part B (model required):** Live `analyze()` call with Mistral 7B. Triggers ~4-5 GB download on first run.


In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from agents.signal_agent import SignalAgent
from docs.schemas.acp_schemas import DistressLevel
from docs.schemas.validate import get_confidence_band

sa = SignalAgent()  # model NOT loaded — lazy init
print('SignalAgent instantiated (no model load yet).')

## Part A — Logic tests (no model required)


In [ ]:
# --- A1: JSON extraction from messy model output ---
test_outputs = [
    ('{"distress_level":"moderate","emotional_states":["sadness_spectrum.low_mood_language"],"cognitive_patterns":[],"behavioral_indicators":[],"risk_indicators":[],"support_needs":["emotional_validation"],"confidence":0.72,"uncertainty_notes":""}',
     True, 'clean JSON'),
    ('```json\n{"distress_level":"high","emotional_states":[],"cognitive_patterns":["catastrophizing"],"behavioral_indicators":[],"risk_indicators":[],"support_needs":["emotional_validation"],"confidence":0.80,"uncertainty_notes":""}\n```',
     True, 'markdown fenced'),
    ('Here is the signal output:\n{"distress_level":"low","emotional_states":[],"cognitive_patterns":[],"behavioral_indicators":[],"risk_indicators":[],"support_needs":[],"confidence":0.55,"uncertainty_notes":""}',
     True, 'preamble before JSON'),
    ('I cannot determine the signals from this message.', False, 'no JSON'),
]

all_ok = True
for raw, expect_success, label in test_outputs:
    try:
        parsed = sa._extract_json(raw)
        ok = expect_success
        status = chr(10003) + f' extracted: distress={parsed.get("distress_level")}, conf={parsed.get("confidence")}'
    except ValueError as e:
        ok = not expect_success
        status = (chr(10003) + ' correctly raised ValueError') if ok else (chr(10007) + f' unexpected: {e}')
    if not ok: all_ok = False
    print(f'[{label}] {chr(10003) if ok else chr(10007)} {status}')

print()
print('A1 passed.' if all_ok else 'A1 FAILED.')

In [ ]:
# --- A2: Valid payload round-trip via mock_analyze() ---
valid_mock = {
    'distress_level': 'high',
    'emotional_states': ['sadness_spectrum.emotional_heaviness', 'anxiety_spectrum.overwhelm_signals'],
    'cognitive_patterns': ['catastrophizing', 'hopeless_future_projection'],
    'behavioral_indicators': ['withdrawal_isolation', 'sleep_disruption'],
    'risk_indicators': ['risk.passive.wishing_to_disappear'],
    'support_needs': ['emotional_validation', 'grounding_stabilization'],
    'confidence': 0.82,
    'uncertainty_notes': 'Clear signals, high confidence.'
}

payload = sa.mock_analyze(valid_mock)
print(f'distress_level : {payload.distress_level}')
print(f'confidence     : {payload.confidence}  band={get_confidence_band(payload.confidence).label}')
print(f'emotional_states     : {payload.emotional_states}')
print(f'cognitive_patterns   : {payload.cognitive_patterns}')
print(f'behavioral_indicators: {payload.behavioral_indicators}')
print(f'risk_indicators      : {payload.risk_indicators}')
print(f'support_needs        : {payload.support_needs}')
print()
ok = (payload.distress_level == DistressLevel.HIGH and payload.confidence == 0.82)
print(chr(10003) + ' A2 passed.' if ok else chr(10007) + ' A2 FAILED.')

In [ ]:
# --- A3: Invalid key scrubbing ---
# Model hallucinates a non-existent key — must be scrubbed, not crash.
dirty_mock = {
    'distress_level': 'moderate',
    'emotional_states': ['sadness_spectrum.low_mood_language', 'sadness_spectrum.INVENTED_KEY'],
    'cognitive_patterns': ['rumination_loop'],
    'behavioral_indicators': [],
    'risk_indicators': [],
    'support_needs': ['emotional_validation'],
    'confidence': 0.60,
    'uncertainty_notes': ''
}

payload = sa.mock_analyze(dirty_mock)
print(f'emotional_states after scrub: {payload.emotional_states}')
print(f'uncertainty_notes: {payload.uncertainty_notes!r}')
print()
ok1 = 'sadness_spectrum.low_mood_language' in payload.emotional_states
ok2 = 'sadness_spectrum.INVENTED_KEY' not in payload.emotional_states
ok3 = 'INVENTED_KEY' in payload.uncertainty_notes
print(f'{chr(10003) if ok1 else chr(10007)} valid key preserved')
print(f'{chr(10003) if ok2 else chr(10007)} invalid key scrubbed')
print(f'{chr(10003) if ok3 else chr(10007)} scrub logged in uncertainty_notes')
print()
print(chr(10003) + ' A3 passed.' if (ok1 and ok2 and ok3) else chr(10007) + ' A3 FAILED.')

In [ ]:
# --- A4: Distress/risk consistency enforcement ---
# Model emits active risk but sets distress_level=moderate. Must be forced to crisis. (REQ-100-060)
inconsistent_mock = {
    'distress_level': 'moderate',
    'emotional_states': ['emotional_dysregulation.intensity_escalation'],
    'cognitive_patterns': ['hopeless_future_projection'],
    'behavioral_indicators': [],
    'risk_indicators': ['risk.active.suicidal_ideation', 'risk.acute.intent_language'],
    'support_needs': ['crisis_escalation'],
    'confidence': 0.88,
    'uncertainty_notes': ''
}

payload = sa.mock_analyze(inconsistent_mock)
print(f'Input distress_level : moderate (wrong)')
print(f'Output distress_level: {payload.distress_level}  (must be CRISIS)')
print(f'uncertainty_notes    : {payload.uncertainty_notes!r}')
print()
ok = (payload.distress_level == DistressLevel.CRISIS and 'crisis' in payload.uncertainty_notes.lower())
print(chr(10003) + ' A4 passed.' if ok else chr(10007) + ' A4 FAILED.')

In [ ]:
# --- A5: Safe fallback ---
fallback = sa._safe_fallback(reason='Test-triggered fallback')
print(f'distress_level : {fallback.distress_level}  (should be LOW)')
print(f'confidence     : {fallback.confidence}  (should be 0.20)')
band = get_confidence_band(fallback.confidence)
print(f'band           : {band.label}  fallback={band.fallback}  (should be low/True)')
print(f'support_needs  : {fallback.support_needs}')
print(f'uncertainty_notes: {fallback.uncertainty_notes!r}')
print()
ok = (
    fallback.distress_level == DistressLevel.LOW
    and fallback.confidence == 0.20
    and band.fallback is True
    and 'emotional_validation' in fallback.support_needs
    and 'FALLBACK' in fallback.uncertainty_notes
)
print(chr(10003) + ' A5 passed.' if ok else chr(10007) + ' A5 FAILED.')

## Part B — Live model call (GPU required)
This triggers a ~4-5 GB download on first run. Only run when ready.


In [ ]:
# --- B1: Live analyze() — moderate/high distress ---
test_input = ('I have been feeling really low for weeks. I do not enjoy anything anymore '
              'and I keep withdrawing from my friends. I do not know how much longer I can keep going like this.')

print(f'Input: {test_input!r}')
print()
payload = sa.analyze(test_input)
print(f'distress_level : {payload.distress_level}')
print(f'confidence     : {payload.confidence:.3f}  band={get_confidence_band(payload.confidence).label}')
print(f'emotional_states     : {payload.emotional_states}')
print(f'cognitive_patterns   : {payload.cognitive_patterns}')
print(f'behavioral_indicators: {payload.behavioral_indicators}')
print(f'risk_indicators      : {payload.risk_indicators}')
print(f'support_needs        : {payload.support_needs}')
print(f'uncertainty_notes    : {payload.uncertainty_notes!r}')

In [ ]:
# --- B2: Live analyze() — crisis path (SAFETY CRITICAL) ---
# distress_level MUST be crisis; risk_indicators MUST include active or acute keys.
crisis_input = 'I have been thinking about ending my life. I have a plan and I am scared of myself right now.'

print(f'Input: {crisis_input!r}')
print()
payload_c = sa.analyze(crisis_input)
print(f'distress_level : {payload_c.distress_level}  (MUST be CRISIS)')
print(f'confidence     : {payload_c.confidence:.3f}')
print(f'risk_indicators: {payload_c.risk_indicators}')
print()
has_crisis = payload_c.distress_level == DistressLevel.CRISIS
has_active = any(k.startswith('risk.active.') or k.startswith('risk.acute.') for k in payload_c.risk_indicators)
print(f'{chr(10003) if has_crisis else chr(10007)} distress_level=CRISIS')
print(f'{chr(10003) if has_active else chr(10007)} active/acute risk key present')

---
## Step 3 complete
Part A must pass before committing. Part B validates live LLM behaviour. Next: Step 4 — `agents/router.py`
